%md
# 02 Bronze ingestion
Loads the 8 raw Olist CSV files into Delta tables in `olist.bronze`, adding _ingested_at and _source_file columns.

In [0]:
from pyspark.sql import functions as F

OPCIONES_CSV = {
    "header": True,
    "multiLine": True,
    "quote": '"',
    "escape": '"',
}
volume_path = "/Volumes/olist/bronze/raw_data"

files = [
    "olist_customers_dataset.csv",
    "olist_order_items_dataset.csv",
    "olist_order_payments_dataset.csv",
    "olist_order_reviews_dataset.csv",
    "olist_orders_dataset.csv",
    "olist_products_dataset.csv",
    "olist_sellers_dataset.csv",
    "product_category_name_translation.csv",
]

exceptions = {
    "product_category_name_translation.csv": "category_translation"
}
def to_table_name(file_name):
    if file_name in exceptions:
        return exceptions[file_name]
    
    name = file_name.replace("olist_", "")
    name = name.replace("_dataset.csv", "")
    return name


tables = {}
for file in files:
    tables[file] = to_table_name(file)

for file, table in tables.items():
    print(f"{file} -> olist.bronze.{table}")

In [0]:
for file_name in files:
    file_path = f"{volume_path}/{file_name}"
    table_name = f"olist.bronze.{to_table_name(file_name)}"

    print(f"Reading {file_path} -> writing {table_name}")

    df = (
        spark.read.options(**OPCIONES_CSV).csv(file_path)
        .withColumn("_ingested_at", F.current_timestamp())
        .withColumn("_source_file", F.col("_metadata.file_name"))
    )
    df.write.mode("overwrite").saveAsTable(table_name)

In [0]:
results = []

for file_name in files:
    file_path = f"{volume_path}/{file_name}"
    table_name = f"olist.bronze.{to_table_name(file_name)}"

    csv_count = spark.read.options(**OPCIONES_CSV).csv(file_path).count()
    table_count = spark.table(table_name).count()

    status = "OK" if csv_count == table_count else "MISMATCH"
    print(f"{table_name}: csv={csv_count} table={table_count} {status}")
    results.append(csv_count == table_count)

assert all(results), "Row counts do not match between CSV files and bronze tables"

In [0]:
CONTEOS_ESPERADOS = {
    "customers": 99441,
    "orders": 99441,
    "order_items": 112650,
    "order_payments": 103886,
    "products": 32951,
    "sellers": 3095,
    "category_translation": 71,
    "order_reviews": 99224,
}
for tabla, esperado in CONTEOS_ESPERADOS.items():
    real = spark.table(f"olist.bronze.{tabla}").count()
    assert real == esperado, f"{tabla}: esperado={esperado}, real={real}"

n_invalidos = spark.sql("""
    SELECT COUNT(*) FROM olist.bronze.order_reviews
    WHERE review_score IS NULL
       OR review_score NOT IN ('1','2','3','4','5')
""").first()[0]
assert n_invalidos == 0, f"order_reviews: {n_invalidos} puntuaciones fuera de 1-5"

LARGO_ID = 32
n_largo = spark.sql(f"""
    SELECT COUNT(*) FROM olist.bronze.order_reviews
    WHERE order_id IS NULL OR LENGTH(order_id) <> {LARGO_ID}
""").first()[0]
assert n_largo == 0, f"order_reviews: {n_largo} order_id con largo distinto de {LARGO_ID}"

print("Bronze verificada: conteos y dominios correctos")